# Apply masks

In [ ]:
from pathlib import Path
import sys
import numpy as np
src_path = str(Path.cwd().parent.parent)
if src_path not in sys.path:
    sys.path.append(src_path)
from src.d01_init_proc.applymask import apply_ROI_masks
from src.d01_init_proc.vis_and_rescale import generate_subtractbg_fig, show_figs
import src.d00_utils.dirnames as dn
import src.d00_utils.utilities as utils
import pandas as pd

from bioio import BioImage
import bioio_ome_tiff
from bioio.writers import OmeTiffWriter
import bioio_tifffile
from skimage.morphology import remove_small_objects
from scipy.ndimage import binary_fill_holes
from skimage.draw import polygon2mask
from matplotlib import pyplot as plt
import numpy.ma as ma
import copy
from tqdm import tqdm

import seaborn as sns

In [ ]:
input_dirpath = Path(input())

In [ ]:
orig_dirpath = Path(input())

In [ ]:
imgpaths_edited = [p for p in input_dirpath.glob('*.ome.tif')]
imgpaths_orig = [p for p in orig_dirpath.glob('*.ome.tif')]
print(len(imgpaths_orig))

imgpaths = np.concatenate([imgpaths_edited, imgpaths_orig])
imgpaths.sort()    
print(len(imgpaths))

In [ ]:
def compute_areas(img, df_idc, df, pixel_area):
    seg_cmpch = 0
    seg_caaxch = 1
    
    regionareas = np.count_nonzero(img, axis=(3, 4)) * pixel_area
    
    df.loc[df_idc, 'compacted area'] = regionareas[:, seg_cmpch, :]
    df.loc[df_idc, 'CAAX-positive area'] = regionareas[:, seg_caaxch, :]
    
    return df

def compute_int(img, df_idc, df):
    
    nonzero_img = np.ma.masked_array(img, img==0)
    mean_int = np.ma.mean(nonzero_img, axis=(3, 4)).squeeze()
    df.loc[df_idc, 'mean actin intensity'] = mean_int

    return df

# def compute_int(img, df_idc, df):
    
#     nonzero_img = np.ma.masked_array(img, img==0)
#     mean_int = np.ma.mean(nonzero_img, axis=(0, 2, 3, 4))
#     df.loc[df_idc, 'cell ch mean int'] = mean_int[cellch]
#     df.loc[df_idc, 'caax ch mean int'] = mean_int[caaxch]
    
#     median_int = np.ma.median(nonzero_img, axis=(0, 2, 3, 4))
#     df.loc[df_idc, 'cell ch median int'] = median_int[cellch]
#     df.loc[df_idc, 'caax ch median int'] = median_int[caaxch]

#     return df

In [ ]:
df = pd.DataFrame()
df['imgpath'] = [p for p in imgpaths]
df['image name'] = [Path(p).name for p in df['imgpath']]
df['image dirpath'] = [Path(p).parent for p in df['imgpath']]
df['edited'] = np.where(df['image dirpath']==input_dirpath, True, False)

df.head()

In [ ]:
seg_cmpch = -2
seg_caaxch = -1

for imgpath in tqdm(imgpaths):
    imgpath = Path(imgpath)
    
    if imgpath.is_file():
        
        # open image and get subset of channels to mask
        img_file = BioImage(imgpath, reader=bioio_tifffile.Reader)
        img = img_file.data
        pixel_area = img_file.physical_pixel_sizes.X * img_file.physical_pixel_sizes.Y

        df_idc = (df['imgpath']==imgpath)
        # df = compute_areas(img[:, -2:, :, :, :], df_idc, df, pixel_area)
        df = compute_int(img[:, 2, np.newaxis, :, :, :], df_idc, df)

print('Done!')

In [ ]:
df['% compaction'] = df['compacted area'] / (df['compacted area'] + df['CAAX-positive area'])
df.head()
df.to_csv(input_dirpath / 'compaction.csv')

In [ ]:
df.at[0, 'image name']

In [ ]:
df['scene_ROI'] = df['image name'].str.split('_').str[5] + '_' + df['image name'].str.split('.ome.tif').str[0].str.split('_').str[7]

imgname_splits = df['image name'].str.replace('-', '_').str.split('_')

df['wellID'] = imgname_splits.str[0] + '-' + imgname_splits.str[6]

In [ ]:
df_subset = df
sns.swarmplot(data=df_subset, x='scene_ROI', y='% compaction', size=2, hue='wellID')
plt.ylim(0, 1)
plt.xticks(rotation=90)

In [ ]:
corr_diff = df.groupby('scene_ROI', sort=False)['% compaction'].diff(periods=1)
corr_diff

In [ ]:
df.loc[corr_diff.abs().idxmax(), 'scene_ROI']

In [ ]:
perc_diff = corr_diff / df['% compaction']
perc_diff

In [ ]:
df_subset = df[df['edited']==False]

sns.swarmplot(data=df_subset, x='wellID', y='% compaction', size=2, hue='wellID')
plt.ylim(0, 1)
plt.xticks(rotation=90)

In [ ]:
df_subset = df[df['edited']==True]
df_subset = df['scP66' in df['image name']]
sns.swarmplot(data=df_subset, x='wellID', y='% compaction', size=2, hue='wellID')
plt.ylim(0, 1)
plt.xticks(rotation=90)

In [ ]:
from scipy import stats


y_label = '% compaction'
statistic, pvalue = stats.ttest_ind(df.loc[df['wellID']=='CE027-A2', '% compaction'].values, df.loc[df['wellID']=='CE027-A4', '% compaction'].values)
p_value = str(float(round(pvalue, 3)))
print(p_value)

print(df.loc[df['wellID']=='CE027-A2', '% compaction'].values.mean())
print(df.loc[df['wellID']=='CE027-A4', '% compaction'].values.mean())

In [ ]:
df.loc[df['wellID']=='CE027-A2', '% compaction'].values

In [ ]:

df = df[df['edited']==True]
sns.swarmplot(data=df, x='mean actin intensity', y='% compaction', size=2, hue='wellID')
plt.show()

sns.swarmplot(data=df, x='wellID', y='mean actin intensity', size=2, hue='wellID')
plt.show()

df_subset = df[df['wellID']=='CE027-A2']
sns.swarmplot(data=df_subset, x='mean actin intensity', y='% compaction', size=2, hue='wellID')
plt.show()


In [ ]:
df